## Week 2 Day 2 - Extra

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs
4. Standarization of outputs with Structured Outputs and define error-cases with Guardrails

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, input_guardrail, tool_output_guardrail, output_guardrail, GuardrailFunctionOutput
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
from pydantic import BaseModel

In [74]:
load_dotenv(override=True)

True

# Sales Agents

In this section we do:

1. Sales Agent Creation
2. Sales Agents conversion into Tools

In [ ]:
instructions1 = "You are a sales agent working for a company, which name will be given by the user, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for which name will be given by the user, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for which name will be given by the user, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [76]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [ ]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

sales_tools = [tool1, tool2, tool3]

# Mail Parser and Sender 

In this section the following is done:

1. Create an agent to format the subject and another to format the body of the mail
2. Convert those agents into tools for using it in the Email Parser Manager

In [78]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [79]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("aortegablasi@gmail.com")  # Change to your verified sender
    to_email = To("aortegablasi@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [80]:
email_sender_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_sender_tools,
    handoff_description="Convert an email to HTML and send it",
    model="gpt-4o-mini")


# Sales Manager

We conclude the lab exercise by adding the following:

1. The Sales Manager who drafts the mails using the Sales Agents as tools
2. Handles the mail to the emailer_agent who is in charge of parsing and sending the mail
3. Uses guardrails and structured outputs to ensure the answers are given in a proper way

In [ ]:
class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent1 = Agent( 
    name="Name check",
    instructions="Check if the user is including someone's personal name in what they want you to do.",
    output_type=NameCheckOutput,
    model="gpt-4o-mini"
)

class CompanyNameCheckOutput(BaseModel):
    is_incorrect_company_name_in_message: bool
    company_name: str

guardrail_agent2 = Agent( 
    name="Company name check",
    instructions="Check if the reference company in the mail is other than 'ComplAI'.",
    output_type=CompanyNameCheckOutput,
    model="gpt-4o-mini"
)

In [ ]:
@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent1, message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info={"found_name": result.final_output},tripwire_triggered=is_name_in_message)

@input_guardrail
async def guardrail_against_company_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent2, message, context=ctx.context)
    is_incorrect_company_name_in_message = result.final_output.is_incorrect_company_name_in_message
    return GuardrailFunctionOutput(output_info={"Company name not matching": result.final_output},tripwire_triggered=is_incorrect_company_name_in_message)

    

In [ ]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=sales_tools,
    handoffs=[emailer_agent],
    input_guardrails = [guardrail_against_name,guardrail_against_company_name],
    model="gpt-4o-mini")

In [ ]:
message = "Send out a cold sales email addressed to Dear CEO from the WorkAI company"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!